# Lecture 3 Demo: Measurement, Observables, and the Quantum Toolbox

Computational companion notebook for Lecture 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram

import pennylane as qml

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"Qiskit version: {__import__('qiskit').__version__}")
print(f"PennyLane version: {qml.__version__}")

## Demo 1. Shot statistics and convergence

Every qubit starts in $\lvert 0\rangle$ by default.
Apply $R_y(\theta)$ and estimate $P(1)$ with increasing shots.

In [ ]:
theta = np.pi / 3
true_p1 = np.sin(theta / 2) ** 2

qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()

sampler = StatevectorSampler()
shots_list = [16, 64, 256, 1024, 4096]
p1_hat = []

for shots in shots_list:
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    est = counts.get("1", 0) / shots
    p1_hat.append(est)
    print(f"shots={shots:4d}  counts={counts}  p1_hat={est:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(shots_list, p1_hat, "o-", lw=2, label="estimate")
ax.axhline(true_p1, color="red", ls="--", label=f"true p1={true_p1:.4f}")
ax.set_xscale("log", base=2)
ax.set_xlabel("shots")
ax.set_ylabel("P(1)")
ax.set_title("Shot-based convergence of measurement probabilities")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Demo 1b. Overlaps, eigenvalues, and what an "observable" is

An **observable** is a Hermitian operator $O$.

- It has an orthonormal eigenbasis $\{(\lambda_i,\lvert\phi_i\rangle)\}$ and a spectral decomposition
  $O=\sum_i \lambda_i\lvert\phi_i\rangle\langle\phi_i\rvert$.
- A projective measurement of $O$ returns one eigenvalue $\lambda_i$ and (in the idealized model) leaves the system in the corresponding eigenstate $\lvert\phi_i\rangle$.

If the system is in state $\lvert\psi\rangle$ before measurement, the **Born rule** says
\[
P(\lambda_i)=\lvert\langle\phi_i\vert\psi\rangle\rvert^2.
\]
For real unit vectors, $\lvert\langle\phi_i\vert\psi\rangle\rvert=\cos\theta$, so this becomes $P=\cos^2\theta$.

Convention: bras are conjugate transposes, $\langle\psi\vert=(\lvert\psi\rangle)^\dagger$. In NumPy, `np.vdot(a, b)` computes $a^\dagger b$.

Finally, the **expectation value** is the average of measurement outcomes:
\[
\langle O\rangle = \langle\psi\vert O\vert\psi\rangle = \sum_i \lambda_i P(\lambda_i).
\]
It is a number (a mean), not a state.

In [ ]:
# We stay in the statevector setting to make the linear algebra explicit.
# Qiskit measurements are in the computational (Z) basis; other bases are
# implemented by a basis-change unitary before measuring.

theta = np.pi / 3

qc_state = QuantumCircuit(1)
qc_state.ry(theta, 0)

sv = Statevector.from_instruction(qc_state)
psi = sv.data.reshape((2,))

ket0 = np.array([1.0, 0.0], dtype=complex)
ket1 = np.array([0.0, 1.0], dtype=complex)
ket_plus = (ket0 + ket1) / np.sqrt(2)
ket_minus = (ket0 - ket1) / np.sqrt(2)

# Z measurement: eigenpairs ( +1, |0> ), ( -1, |1> )
p0 = np.abs(np.vdot(ket0, psi)) ** 2
p1 = np.abs(np.vdot(ket1, psi)) ** 2

print("State |psi>:", psi)
print(f"Z-basis overlaps: P(0)={p0:.6f}, P(1)={p1:.6f}, sum={p0+p1:.6f}")

Z = np.array([[1, 0], [0, -1]], dtype=complex)
exp_z_from_probs = (+1) * p0 + (-1) * p1
exp_z_direct = np.vdot(psi, Z @ psi)
print(f"<Z> from probs:   {exp_z_from_probs:.6f}")
print(f"<Z> as <psi|Z|psi>: {np.real(exp_z_direct):.6f}")

# X measurement: eigenpairs ( +1, |+> ), ( -1, |-> )
p_plus = np.abs(np.vdot(ket_plus, psi)) ** 2
p_minus = np.abs(np.vdot(ket_minus, psi)) ** 2

X = np.array([[0, 1], [1, 0]], dtype=complex)
exp_x_from_probs = (+1) * p_plus + (-1) * p_minus
exp_x_direct = np.vdot(psi, X @ psi)

print(f"X-basis overlaps: P(+ )={p_plus:.6f}, P(-)={p_minus:.6f}, sum={p_plus+p_minus:.6f}")
print(f"<X> from probs:   {exp_x_from_probs:.6f}")
print(f"<X> as <psi|X|psi>: {np.real(exp_x_direct):.6f}")

# Simulate an X-basis measurement by rotating X-eigenstates to Z-eigenstates.
# H|+> = |0>, H|-> = |1>.
qc_x_meas = QuantumCircuit(1)
qc_x_meas.ry(theta, 0)
qc_x_meas.h(0)
qc_x_meas.measure_all()

counts_x = sampler.run([qc_x_meas], shots=4096).result()[0].data.meas.get_counts()
phat_plus = counts_x.get("0", 0) / 4096  # corresponds to |+>
phat_minus = counts_x.get("1", 0) / 4096  # corresponds to |->

print("X-basis measurement via H then measure:", counts_x)
print(f"Empirical P(+ )≈{phat_plus:.6f}, P(-)≈{phat_minus:.6f}")

## (Moved) SWAP test and fidelity

This material is used in the kernel storyline and has been moved to `lectures/lecture_05/lecture_05_demo.ipynb`.

In [ ]:
# Moved to lectures/lecture_05/lecture_05_demo.ipynb
# (kept out of Lecture 3 to reduce cognitive load).


## (Moved) Mixed states and noise

This material is now in `lectures/lecture_13/lecture_13_demo.ipynb` (Noise, mixed states, and error mitigation).

In [ ]:
# Moved to lectures/lecture_13/lecture_13_demo.ipynb
# (kept out of Lecture 3 to reduce cognitive load).


## Demo 2. Bell-state observables

Construct $\lvert\Phi^+\rangle = (\lvert00\rangle + \lvert11\rangle)/\sqrt{2}$ and evaluate selected expectation values.

In [ ]:
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

estimator = StatevectorEstimator()
observables = {
    "ZZ": SparsePauliOp("ZZ"),
    "XX": SparsePauliOp("XX"),
    "ZI": SparsePauliOp("ZI"),
}

for name, op in observables.items():
    pub = (qc_bell, op)
    val = estimator.run([pub]).result()[0].data.evs
    print(f"<{name}> = {float(np.real(val)):.6f}")

## Demo 3. Sampler vs estimator on the same state

Sampler returns a distribution over bitstrings.
Estimator returns the expectation value of a chosen observable.

In [ ]:
theta = 0.9

# Sampler path (needs measurements)
qc_s = QuantumCircuit(1)
qc_s.ry(theta, 0)
qc_s.measure_all()
counts = sampler.run([qc_s], shots=1024).result()[0].data.meas.get_counts()
print("Sampler counts:", counts)

# Estimator path (no measurements)
qc_e = QuantumCircuit(1)
qc_e.ry(theta, 0)
ev_z = estimator.run([(qc_e, SparsePauliOp("Z"))]).result()[0].data.evs
print(f"Estimator <Z>: {float(np.real(ev_z)):.6f}  (expected cos(theta)={np.cos(theta):.6f})")

plot_histogram(counts)

## Demo 4. Real-data mini experiment: Iris with quantum-derived features

Binary task: versicolor vs virginica using two petal features.

We compare:
1. logistic regression on standardized raw features
2. logistic regression on quantum-derived features $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ from a one-qubit encoding.

In [ ]:
iris = load_iris()
X_all = iris.data[:, 2:4]  # petal length, petal width
y_all = iris.target

# Keep classes 1 and 2 only
mask = y_all != 0
X = X_all[mask]
y = (y_all[mask] == 2).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Map standardized values to angles in [0, pi]
def to_angles(X_in):
    X_min = X_train_s.min(axis=0)
    X_max = X_train_s.max(axis=0)
    denom = np.where((X_max - X_min) < 1e-12, 1.0, X_max - X_min)
    X_norm = (X_in - X_min) / denom
    X_clip = np.clip(X_norm, 0.0, 1.0)
    return np.pi * X_clip

A_train = to_angles(X_train_s)
A_test = to_angles(X_test_s)

In [ ]:
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def one_qubit_features(a0, a1):
    qml.RY(a0, wires=0)
    qml.RZ(a1, wires=0)
    return (
        qml.expval(qml.PauliX(0)),
        qml.expval(qml.PauliY(0)),
        qml.expval(qml.PauliZ(0)),
    )


def quantum_feature_matrix(A):
    feats = [one_qubit_features(a0, a1) for a0, a1 in A]
    return np.array(feats, dtype=float)

Q_train = quantum_feature_matrix(A_train)
Q_test = quantum_feature_matrix(A_test)

clf_raw = LogisticRegression(max_iter=2000)
clf_raw.fit(X_train_s, y_train)
pred_raw = clf_raw.predict(X_test_s)

clf_q = LogisticRegression(max_iter=2000)
clf_q.fit(Q_train, y_train)
pred_q = clf_q.predict(Q_test)

acc_raw = accuracy_score(y_test, pred_raw)
acc_q = accuracy_score(y_test, pred_q)

print(f"Raw-feature logistic regression accuracy:      {acc_raw:.3f}")
print(f"Quantum-derived feature accuracy (X,Y,Z):      {acc_q:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(X_test_s[:, 0], X_test_s[:, 1], c=y_test, cmap="coolwarm", s=35)
axes[0].set_title("Standardized raw features")
axes[0].set_xlabel("feature 1")
axes[0].set_ylabel("feature 2")
axes[0].grid(True, alpha=0.2)

axes[1].scatter(Q_test[:, 0], Q_test[:, 1], c=y_test, cmap="coolwarm", s=35)
axes[1].set_title("Quantum-derived features: <X>, <Y>")
axes[1].set_xlabel(r"$\langle X\rangle$")
axes[1].set_ylabel(r"$\langle Y\rangle$")
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()